# Joining model with ad exposure

Each block starting with a percent-marker comment is one notebook cell, in notebook order.
Assumes the imports and `panel` from your cell 1 (np, pd, smf, stats, expit, u, CUT).

## [setup] fit_logit -- skip if you already define it elsewhere

In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import optimize, stats
from scipy.special import expit, logit

import utils as u

u.apply_style()
CUT = u.CUT_T                                        # spring 2019
BLUE, ORANGE = '#2a78d6', '#eb6834'

panel = u.load_panel(
    'data/customer_panel.csv', 
    extra_cols=('customer_id', 'discount_flag', 'search_flag','social_ad_impressions', 'podcast_ad_listen'))

In [4]:
def fit_logit(formula, data):
    """Logit with standard errors clustered by customer."""
    return smf.logit(formula, data).fit(disp=0, cov_type='cluster',
                                        cov_kwds={'groups': data['customer_id']})

## [section 1] Sample -- replaces the current sample cell

In [5]:
SCORES = ['tv', 'radio', 'social', 'podcast']
CHANNELS = {'pod': ('podcast_ad_listen', 'podcast', 'podcast_spend'),      # name: (panel column, score, spend)
            'soc': ('social_ad_impressions', 'social', 'social_spend')}
SEASON_ORDER = {'winter': 0, 'spring': 1, 'summer': 2, 'fall': 3}          # t = 0 is winter 2017

df = panel.sort_values(['customer_id', 't']).reset_index(drop=True)
member = df.groupby('customer_id')['sign_up_flag']
df['prev_member'] = member.shift(1)
df['ever_member'] = member.cummax().groupby(df['customer_id']).shift(1)
df['score_year'] = np.where(df['t'].ge(3), 2017 + (df['t'] - 3) // 4, -1)       # summer refresh applies from fall

# Exposure history, built on every customer-season (members too) so the lags are complete.
# Everything here is known before season t starts.
for ch, (col, _, _) in CHANNELS.items():
    reached = df[col].gt(0).astype(float)
    by_cust = reached.groupby(df['customer_id'])
    df[f'{ch}_lag1'] = by_cust.shift(1)                                         # reached last season
    df[f'{ch}_lag2'] = by_cust.shift(2)
    df[f'{ch}_share'] = (by_cust.cumsum() - reached) / df['t'].clip(lower=1)    # share of earlier seasons reached

keep = df['region_name'].ne(u.CONTROL) & df['t'].ge(3) & df['prev_member'].eq(0)
scores = (pd.read_csv('data/customer_propensities.csv',
                      usecols=['customer_id', 'year', *[f'customer_{s}_score' for s in SCORES]])
            .rename(columns={'year': 'score_year', **{f'customer_{s}_score': s for s in SCORES}}))
spend = pd.read_csv('data/aggregate.csv')                                       # planned media spend by region-season
spend['t'] = (spend['year'] - 2017) * 4 + spend['season'].map(SEASON_ORDER)
spend = spend[['region_name', 't', 'podcast_spend', 'social_spend']]

d = (df[keep].merge(scores, on=['customer_id', 'score_year'], how='left', validate='many_to_one')
             .merge(spend, on=['region_name', 't'], how='left', validate='many_to_one')
             .rename(columns={'sign_up_flag': 'joined', 'membership_price': 'price', 'discount_flag': 'offer',
                              'podcast_ad_listen': 'pod', 'social_ad_impressions': 'soc'})
             .reset_index(drop=True))
d['ever_member'] = d['ever_member'].astype(int)
d['post'] = d['t'].ge(CUT).astype(int)
d['refresh'] = d['t'].mod(4).eq(3).astype(int)                                  # fall: first season on new scores
d['depth'] = np.where(d['offer'].eq(1), np.where(d['post'].eq(1), 12, 15) - d['price'], 0)
del panel, df


def add_terms(x):
    """Price and exposure terms of the joining model. Used for the data and for the prediction grid."""
    x = x.copy()
    x['price_sq'] = (x['price'] - 12) ** 2
    x['pod_soc'] = x['pod'] * x['soc']
    x['sqrt_pod_soc'] = np.sqrt(x['pod_soc'])
    x['pod_zero'] = x['pod'].eq(0).astype(int)
    x['soc_zero'] = x['soc'].eq(0).astype(int)
    return x


d = add_terms(d)
HISTORY = [f'{ch}_{k}' for ch in CHANNELS for k in ('lag1', 'lag2', 'share')]
assert d[[*SCORES, *HISTORY, 'podcast_spend', 'social_spend']].notna().all().all()
print(f"{len(d):,} rows, {d['customer_id'].nunique():,} customers, joining rate {d['joined'].mean():.1%}, "
      f"{d['offer'].mean():.0%} offered")

# Why exposure belongs in the model: joining rate (%) by price and exposure
exposure_band = pd.cut(d['pod_soc'], [-1, 0, 25, 80, 200, np.inf],
                       labels=['none', '1-25', '26-80', '81-200', '200+'])
display((100 * d.pivot_table(index=exposure_band, columns='price', values='joined', observed=True)).round(1)
        .rename_axis('listens x impressions'))

2,669,723 rows, 349,623 customers, joining rate 6.3%, 25% offered


price,7,8,9,10,11,12,13,14,15
listens x impressions,,,,,,,,,
none,14.3,9.8,5.8,3.1,1.3,0.5,0.1,0.0,0.0
1-25,35.5,27.7,20.1,12.1,6.5,2.6,0.9,0.2,0.0
26-80,65.3,58.5,48.3,37.9,25.6,14.6,6.7,2.6,0.5
81-200,96.6,95.6,92.5,88.3,81.7,70.7,57.6,39.9,22.8
200+,100.0,100.0,100.0,100.0,100.0,100.0,99.8,98.8,95.9


## [section 2] Model design and checks -- replaces 2.1 and 2.2

In [6]:
EXPOSURE = ['pod', 'soc', 'sqrt_pod_soc', 'pod_soc', 'pod_zero', 'soc_zero']
RHS = 'offer + ' + ' + '.join(EXPOSURE) + ' + C(region_name)'
FORMULA = 'joined ~ price + price_sq + ' + RHS                 # no season effects, no propensity scores

final = fit_logit(FORMULA, d)

# 2.1 Price shape: one dummy per price vs the quadratic
dummies = fit_logit('joined ~ C(price) + ' + RHS, d)
lr = 2 * (dummies.llf - final.llf)
dof = dummies.df_model - final.df_model
print(f"Price dummies vs quadratic: LR = {lr:.1f} on {dof:.0f} df, p = {stats.chi2.sf(lr, dof):.3f}")
step = pd.DataFrame({'price dummies': np.r_[0, dummies.params.filter(like='C(price)').to_numpy()],
                     'quadratic': final.params['price'] * (np.arange(7, 16) - 7)
                                  + final.params['price_sq'] * ((np.arange(7, 16) - 12) ** 2 - 25)},
                    index=range(7, 16))
display(step.round(2).T.rename_axis('log-odds vs $7', axis=1))
del dummies

# 2.2 Are season effects still needed once exposure is in?
with_t = fit_logit(FORMULA + ' + C(t)', d)
season_fx = with_t.params.filter(like='C(t)')
print(f"Season effects with exposure in the model: {season_fx.min():+.2f} to {season_fx.max():+.2f} log-odds; "
      f"log-likelihood {final.llf:,.0f} without them, {with_t.llf:,.0f} with them")
del with_t

# 2.3 Price slope by season (at $12), exposure in the model
seasons = sorted(d['t'].unique())
slope_terms = [f'price_t{k}' for k in seasons]
by_season = d.assign(**{f'price_t{k}': (d['price'] - 12) * d['t'].eq(k) for k in seasons})
fit_s = fit_logit(FORMULA.replace('price + ', ' + '.join(slope_terms) + ' + ') + ' + C(t)', by_season)
slopes = pd.DataFrame({'price slope at $12': fit_s.params[slope_terms].to_numpy(),
                       'joining rate': d.groupby('t')['joined'].mean().to_numpy()},
                      index=[u.period_label(k) for k in seasons])
del fit_s, by_season
display(slopes.round(3).T)

Price dummies vs quadratic: LR = 27.4 on 6 df, p = 0.000


log-odds vs $7,7,8,9,10,11,12,13,14,15
price dummies,0.0,-0.44,-0.99,-1.68,-2.43,-3.27,-4.17,-5.19,-6.46
quadratic,-0.0,-0.45,-1.00,-1.65,-2.40,-3.26,-4.21,-5.27,-6.43


Season effects with exposure in the model: -0.04 to +0.10 log-odds; log-likelihood -223,682 without them, -223,639 with them


,Fall 2017,Win 2018,Spr 2018,Sum 2018,Fall 2018,Win 2019,Spr 2019,Sum 2019,Fall 2019
price slope at $12,-0.856,-0.890,-1.015,-0.883,-0.874,-0.860,-0.911,-0.900,-0.909
joining rate,0.036,0.045,0.010,0.120,0.024,0.086,0.044,0.081,0.136


## [section 3.1] Final model -- replaces 3.1

In [7]:
terms = ['price', 'price_sq', 'offer', *EXPOSURE]
ci = final.conf_int()
display(pd.DataFrame({'estimate': final.params[terms], 'ci_low': ci.loc[terms, 0],
                      'ci_high': ci.loc[terms, 1], 'p_value': final.pvalues[terms]}).round(4))
print(f"Price slope: {final.params['price'] + 6 * final.params['price_sq']:.2f} log-odds per $1 at $15, "
      f"{final.params['price']:.2f} at $12, {final.params['price'] - 10 * final.params['price_sq']:.2f} at $7")

,estimate,ci_low,ci_high,p_value
price,-0.9049,-0.9112,-0.8985,0.0000
price_sq,-0.0507,-0.0521,-0.0493,0.0000
offer,-0.0219,-0.0453,0.0015,0.0666
pod,-0.0001,-0.0022,0.0020,0.9359
soc,-0.0325,-0.0368,-0.0283,0.0000
sqrt_pod_soc,0.0395,0.0286,0.0503,0.0000
pod_soc,0.0437,0.0430,0.0444,0.0000
pod_zero,-0.8647,-0.9063,-0.8231,0.0000
soc_zero,-0.0696,-0.1098,-0.0293,0.0007


Price slope: -1.21 log-odds per $1 at $15, -0.90 at $12, -0.40 at $7


## [section 3.2] Calibration with actual exposure (a fit check, not a forecast) -- replaces 3.2; 3.3 runs unchanged

In [8]:
d['p_hat'] = final.predict(d)
cal = d.groupby(['post', 'offer', 'price']).agg(rows=('joined', 'size'), actual=('joined', 'mean'),
                                                predicted=('p_hat', 'mean'))
cal['gap_pp'] = 100 * (cal['actual'] - cal['predicted'])
print(f"Largest gap: {cal['gap_pp'].abs().max():.2f} pp")
display(cal.round(4))

Largest gap: 0.31 pp


rows  actual  predicted  gap_pp
post offer price                                    
0    0     15     1399080  0.0366     0.0367 -0.0109
     1     10       93398  0.1667     0.1698 -0.3086
           11       93046  0.1312     0.1320 -0.0898
           12       94109  0.0975     0.0977 -0.0270
           13       93666  0.0729     0.0718  0.1042
           14       93232  0.0529     0.0515  0.1416
1    0     12      602384  0.0598     0.0595  0.0254
     1     7        40036  0.2522     0.2516  0.0570
           8        40376  0.2042     0.2030  0.1258
           9        40308  0.1572     0.1565  0.0686
           10       40177  0.1207     0.1183  0.2398
           11       39911  0.0839     0.0846 -0.0749

## [section 4] Forecasting without current-season exposure -- replaces all of section 4

In [9]:
REACH_RHS = '{ch}_lag1 + {ch}_lag2 + {ch}_share + {score} + refresh'


def fit_forecaster(train):
    """Everything the forecast needs, learned from past seasons only."""
    parts = {'join': fit_logit(FORMULA, train), 'reach': {}, 'cap': {}}
    for ch, (_, score, spend_col) in CHANNELS.items():
        y = train.assign(reached=train[ch].gt(0).astype(int))
        parts['reach'][ch] = fit_logit('reached ~ ' + REACH_RHS.format(ch=ch, score=score), y)
        by_t = train.groupby('t').agg(top=(ch, 'max'), spend=(spend_col, 'first'))
        parts['cap'][ch] = (by_t['top'] / by_t['spend']).mean()        # highest exposure per unit of spend
    return parts


def forecast_join(parts, new):
    """P(join) for prospects in a future season, from price, planned spend, scores and exposure history.

    Exposure this season is unknown, so the joining model is averaged over its distribution:
    reached or not (reach models), then evenly spread from 1 up to cap x planned spend.
    """
    new = new.copy()
    for ch in CHANNELS:
        new[f'p_{ch}'] = parts['reach'][ch].predict(new)
    cells = new[['t', 'region_name', 'price', 'offer', 'podcast_spend', 'social_spend']].drop_duplicates()
    rows = []
    for c in cells.itertuples(index=False):
        top_pod = max(int(round(parts['cap']['pod'] * c.podcast_spend)), 1)
        top_soc = max(int(round(parts['cap']['soc'] * c.social_spend)), 1)
        pod, soc = np.meshgrid(np.arange(top_pod + 1), np.arange(top_soc + 1), indexing='ij')
        grid = add_terms(pd.DataFrame({'region_name': c.region_name, 'price': c.price, 'offer': c.offer,
                                       'pod': pod.ravel(), 'soc': soc.ravel()}))
        p = np.asarray(parts['join'].predict(grid)).reshape(pod.shape)
        rows.append({'t': c.t, 'region_name': c.region_name, 'price': c.price, 'offer': c.offer,
                     'j_none': p[0, 0], 'j_soc': p[0, 1:].mean(), 'j_pod': p[1:, 0].mean(), 'j_both': p[1:, 1:].mean()})
    new = new.merge(pd.DataFrame(rows), on=['t', 'region_name', 'price', 'offer'], how='left', validate='many_to_one')
    return ((1 - new['p_pod']) * (1 - new['p_soc']) * new['j_none'] + (1 - new['p_pod']) * new['p_soc'] * new['j_soc']
            + new['p_pod'] * (1 - new['p_soc']) * new['j_pod'] + new['p_pod'] * new['p_soc'] * new['j_both']).to_numpy()


def by_depth(x):
    out = x.groupby(['t', 'depth']).agg(actual=('joined', 'mean'), predicted=('forecast', 'mean'))
    out['gap_pp'] = 100 * (out['actual'] - out['predicted'])
    return out


# 4.1 Out-of-time: fit on the six pre-cut seasons, forecast the three post-cut seasons.
#     Uses no post-cut joins and no current-season exposure. Summer and fall use exposure history
#     through the season before, so each is a one-season-ahead forecast of reach.
pre_parts = fit_forecaster(d[d['post'].eq(0)])
post = d[d['post'].eq(1)].copy()
post['forecast'] = forecast_join(pre_parts, post)
oot = by_depth(post)
print(f"Pre-cut fit: largest post-cut gap {oot['gap_pp'].abs().max():.2f} pp; "
      f"at $9-$12 {oot.loc[oot.index.get_level_values('depth') <= 3, 'gap_pp'].abs().max():.2f} pp")
display((100 * oot[['actual', 'predicted']]).round(1).unstack('depth').rename(index=u.period_label))
reach_check = post.assign(pod_reached=post['pod'].gt(0), soc_reached=post['soc'].gt(0),
                          p_pod=pre_parts['reach']['pod'].predict(post), p_soc=pre_parts['reach']['soc'].predict(post))
display(reach_check.groupby('t')[['pod_reached', 'p_pod', 'soc_reached', 'p_soc']].mean().round(3)
        .rename(index=u.period_label))
del pre_parts, reach_check

# 4.2 Rolling origin: fit on every season before k, forecast season k. This is how the tool will be used.
rolling = []
for k in seasons[3:]:
    test = d[d['t'].eq(k)].copy()
    test['forecast'] = forecast_join(fit_forecaster(d[d['t'].lt(k)]), test)
    rolling.append(test[['t', 'depth', 'joined', 'forecast']])
roll = by_depth(pd.concat(rolling))
print(f"Rolling origin: largest gap {roll['gap_pp'].abs().max():.2f} pp")
display((100 * roll[['actual', 'predicted']]).round(1).unstack('depth').rename(index=u.period_label))

Pre-cut fit: largest post-cut gap 0.98 pp; at $9-$12 0.98 pp


actual                               predicted                    \
depth          0     1     2     3     4     5         0     1     2     3   
t                                                                            
Spr 2019     1.8   3.8   7.3  11.6  16.7  22.5       1.8   4.1   7.4  11.7   
Sum 2019     5.3   8.0  12.1  15.7  20.8  25.3       5.5   8.7  12.3  16.2   
Fall 2019   11.2  13.9  17.2  20.1  23.9  28.1      10.7  13.7  16.2  19.2   

                       
depth         4     5  
t                      
Spr 2019   16.8  22.7  
Sum 2019   21.1  25.8  
Fall 2019  23.1  27.2

,pod_reached,p_pod,soc_reached,p_soc
t,,,,
Spr 2019,0.323,0.339,0.763,0.798
Sum 2019,0.328,0.337,0.800,0.798
Fall 2019,0.309,0.306,0.806,0.803


Rolling origin: largest gap 2.04 pp


actual                               predicted                    \
depth          0     1     2     3     4     5         0     1     2     3   
t                                                                            
Sum 2018     9.9  12.4  14.8  17.4  21.5  24.8       9.1  11.6  14.0  16.6   
Fall 2018    0.9   2.0   4.0   6.2   9.2  13.5       0.9   2.2   4.1   6.7   
Win 2019     6.8   8.8  11.1  13.7  16.2  19.4       6.7   8.7  10.9  13.8   
Spr 2019     1.8   3.8   7.3  11.6  16.7  22.5       1.8   4.1   7.4  11.7   
Sum 2019     5.3   8.0  12.1  15.7  20.8  25.3       5.3   8.2  11.9  16.0   
Fall 2019   11.2  13.9  17.2  20.1  23.9  28.1      11.0  13.8  16.5  19.6   

                       
depth         4     5  
t                      
Sum 2018   19.7  22.8  
Fall 2018   9.7  13.7  
Win 2019   16.4  19.7  
Spr 2019   16.8  22.7  
Sum 2019   21.2  26.0  
Fall 2019  23.6  27.6

## [section 6] What goes into the tool -- replaces the export cell

In [10]:
out = Path('outputs')
out.mkdir(exist_ok=True)
parts = fit_forecaster(d)                                                    # all nine seasons
parts['join'].params.rename('coefficient').to_csv(out / 'join_model_params.csv')
for ch in CHANNELS:
    parts['reach'][ch].params.rename('coefficient').to_csv(out / f'reach_{ch}_params.csv')
pd.Series(parts['cap'], name='max_exposure_per_unit_spend').to_csv(out / 'exposure_caps.csv')
slopes.to_csv(out / 'join_price_slope_by_season.csv')

## Suggested text for the Findings block at the top (numbers from a run on the uploaded data)

* Joining depends on price and on ad exposure together. At \$12, prospects with no podcast listens or social
  impressions join at 0.5%; those with heavy exposure of both join at over 70%.
* Each \$1 lower price raises the log-odds of joining by 0.90 at \$12 (1.21 at \$15, 0.40 at \$7), holding exposure fixed.
  The slope still comes from the randomized discount depths.
* Season effects are not needed once exposure is in the model (-0.04 to +0.10 log-odds), and the price slope
  is the same in every season (-0.86 to -1.02 at \$12).
* With actual exposure the model is within 0.31 pp of actual joining at every price.
* Forecast without current-season exposure (price, planned spend, scores, exposure history): fit on pre-cut
  seasons only, post-cut joining is within 1.0 pp at every price and season; fit on all earlier seasons,
  within 0.7 pp for the three post-cut seasons.
* Limitations: the forecast needs planned podcast and social spend as inputs, and reach is forecast one season ahead.